# 🤖 Week 3: Predictive Modeling & Algorithm Selection
**Author:** Krishnaveni Mitukula 
**Phase:** Machine Learning Implementation

## DAY 1: Data Preparation & Train/Test Split
**Objective:** Load the SCALED dataset generated in Week 1 (which is optimized for Machine Learning). Separate the clinical features (`X`) from the target diagnosis (`y`), and split the data into 80% training data (for the AI to learn) and 20% testing data (to evaluate the AI).

In [10]:
# WEEK 3 - DAY 1: Train/Test Split

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

# 1. Load the SCALED dataset we prepared in Week 1
df = pd.read_csv('data/scaled_heart_disease_data.csv')
print("Scaled ML dataset loaded successfully!")

# --- CLINICAL CLEANING FOR STRINGS --- 
# Replace '?' placeholders (from missing values in raw data) with NaN
df = df.replace('?', np.nan)

# Impute missing values with Mode (since ca and thal are categorical columns)
df['ca'] = df['ca'].fillna(df['ca'].mode()[0])
df['thal'] = df['thal'].fillna(df['thal'].mode()[0])

# Convert all columns (except target & age_group) to float so sklearn can run
for col in df.columns:
    if col not in ['target', 'age_group']:
        df[col] = df[col].astype(float)

# 2. Separate Features (X) and Target (y)
# We drop 'age_group' because it was just for BI dashboards, not for ML
X = df.drop(columns=['target', 'age_group'])
y = df['target']

# 3. Perform 80/20 Train-Test Split (stratified)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

print(f"Training Data: {X_train.shape[0]} patients")
print(f"Testing Data: {X_test.shape[0]} patients")

Scaled ML dataset loaded successfully!
Training Data: 242 patients
Testing Data: 61 patients


## DAY 2: Class Imbalance Assessment & SMOTE
**Objective:** Evaluate if there is a class imbalance in our training target (`y_train`). If one class represents a significant minority (e.g., less than 40% of the data), we will apply SMOTE (Synthetic Minority Over-sampling Technique) to generate synthetic training examples to ensure the model trains without bias.

In [11]:
# ==========================================
# WEEK 3 - DAY 2: Class Imbalance & SMOTE
# ==========================================

# 1. Check current class distribution in training set
class_counts = y_train.value_counts()
class_percentages = y_train.value_counts(normalize=True) * 100

print("--- Training Set Class Distribution ---")
for val, count in class_counts.items():
    label = "Heart Disease" if val == 1 else "Healthy"
    print(f"{label}: {count} patients ({class_percentages[val]:.2f}%)")

# 2. Check if we need SMOTE (threshold: if minority class is < 40%)
minority_percentage = class_percentages.min()

if minority_percentage < 40.0:
    print("\n[Clinical Decision] Class imbalance detected. Applying SMOTE to balance the training data...")
    try:
        from imblearn.over_sampling import SMOTE
    except ImportError:
        print("Installing imbalanced-learn library...")
        import sys
        !{sys.executable} -m pip install -q imbalanced-learn
        from imblearn.over_sampling import SMOTE
        
    smote = SMOTE(random_state=42)
    X_train_res, y_train_res = smote.fit_resample(X_train, y_train)
    print(f"Resampled Training Data Shape: {X_train_res.shape[0]} patients")
    print(f"New Class Distribution:\n{y_train_res.value_counts()}")
else:
    print("\n[Clinical Decision] The classes are well-balanced (minority class is >= 40%).")
    print("SMOTE is not strictly required. We will use the original split to maintain data integrity.")
    # Keep variables named consistently for the next cells
    X_train_res, y_train_res = X_train, y_train

--- Training Set Class Distribution ---
Healthy: 131 patients (54.13%)
Heart Disease: 111 patients (45.87%)

[Clinical Decision] The classes are well-balanced (minority class is >= 40%).
SMOTE is not strictly required. We will use the original split to maintain data integrity.


**Clinical Imbalance Assessment:**
Our training cohort consists of 54.13% healthy patients and 45.87% diseased patients. Because the minority class represents nearly 46% of our data, the dataset is statistically well-balanced. Applying SMOTE unnecessarily could introduce artificial noise into our clinical features, so we will proceed with our original, high-quality train-test split for modeling.

## DAY 3: Baseline Model - Logistic Regression
**Objective:** Train a baseline Logistic Regression model on the training dataset. We will use this simple linear classifier as a benchmark for performance. Clinically, we will analyze the classification report and confusion matrix, focusing on the **Recall** of the Heart Disease class (to minimize missed diagnoses).

In [14]:
# WEEK 3 - DAY 3: Logistic Regression Baseline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, recall_score, roc_auc_score

# 1. Train Logistic Regression
lr_model = LogisticRegression(random_state=42)
lr_model.fit(X_train_res, y_train_res)

# 2. Predict on Test Set
y_pred_lr = lr_model.predict(X_test)
y_prob_lr = lr_model.predict_proba(X_test)[:, 1]

# 3. Evaluate
print("--- Baseline Logistic Regression Performance ---")
print(classification_report(y_test, y_pred_lr))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_lr))
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_prob_lr):.4f}")
print(f"Recall (Heart Disease): {recall_score(y_test, y_pred_lr):.4f}")

--- Baseline Logistic Regression Performance ---
              precision    recall  f1-score   support

           0       0.93      0.82      0.87        33
           1       0.81      0.93      0.87        28

    accuracy                           0.87        61
   macro avg       0.87      0.87      0.87        61
weighted avg       0.88      0.87      0.87        61

Confusion Matrix:
[[27  6]
 [ 2 26]]
ROC-AUC Score: 0.9535
Recall (Heart Disease): 0.9286
